# Deliverable 2 – Campus Coffee Shop
INF201 · Sigurd Aleksander Ruud and Shem Abdul-Rahman

In [ ]:
import json
import re
from pathlib import Path

import numpy as np
import pandas as pd
#imports and gets all the packages needed for the code to run
DATA_DIR = Path("data")
OUT_DIR = Path("output")
OUT_DIR.mkdir(exist_ok=True)

## Task 1.1 Robust file reader
Try UTF-8 first. If it fails, fall back to ISO-8859-1.

In [ ]:
def read_file(path) -> tuple[str, str]:
    """Read a text file as UTF-8, fall back to ISO-8859-1. Returns (text, encoding)."""
    path = Path(path)
    try:
        return path.read_text(encoding="utf-8"), "utf-8"
    except UnicodeDecodeError:
        return path.read_text(encoding="iso-8859-1"), "iso-8859-1"

## Task 1.2 Log parser
Regex captures the 6 fields. Noise lines don't match and are skipped.

In [ ]:
LINE_PATTERN = re.compile(
    r"\[(\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2})\]"   # timestamp
    r" \[(\w+)\]"                                  # register_id
    r" item=(\w+)"                                 # item
    r" price=(\d+\.\d+)"                           # price
    r" ([A-Z]{3})"                                 # currency
    r" \((SUCCESS|REFUND|CANCELLED)\)"             # status
)


def parse_log(text: str) -> list[dict]:
    """Return one dict per valid sales line; noise lines are ignored."""
    records = []
    for line in text.splitlines():
        match = LINE_PATTERN.search(line)
        if match:
            ts, reg, item, price, cur, status = match.groups()
            records.append({
                "timestamp": ts,
                "register_id": reg,
                "item": item,
                "price": float(price),
                "currency": cur,
                "status": status,
            })
    return records

In [ ]:
test = """[2026-10-15 08:30:00] [POS_01] item=Latte price=4.50 EUR (SUCCESS)
--- PRINTER PAPER LOW ---"""
parse_log(test)

[{'timestamp': '2026-10-15 08:30:00',
  'register_id': 'POS_01',
  'item': 'Latte',
  'price': 4.5,
  'currency': 'EUR',
  'status': 'SUCCESS'}]

## Task 2.1 Recursive file search
The function calls itself on every sub-folder (no glob/rglob/os.walk).

In [ ]:
def find_log_files(folder, extension: str = ".log") -> list[Path]:
    """Recursively find all files with the given extension."""
    folder = Path(folder)
    found = []
    for item in folder.iterdir():
        if item.is_dir():
            found.extend(find_log_files(item, extension))   # calls itself
        elif item.is_file() and item.suffix == extension:
            found.append(item)
    return found

In [ ]:
files = find_log_files(DATA_DIR)
len(files), files[0]

(9, PosixPath('data/2026/science_building/POS_01/register_sales.log'))

## Task 2.2 Recursive search for active registers
Handles dicts (check + search values), lists (search elements), and other values (stop).

In [ ]:
def find_active_registers(data) -> set[str]:
    """Return the ids of all registers with "active": true, at any depth."""
    active = set()
    if isinstance(data, dict):
        if data.get("active") is True and "id" in data:
            active.add(data["id"])
        for value in data.values():
            active |= find_active_registers(value)
    elif isinstance(data, list):
        for element in data:
            active |= find_active_registers(element)
    return active

In [ ]:
config = json.loads((DATA_DIR / "store_config.json").read_text(encoding="utf-8"))
find_active_registers(config)

{'POS_01', 'POS_02', 'POS_03'}

## Task 3.1 Outlier removal
A value is an outlier if |x − μ| > 3σ. Outliers are replaced by the median of the valid prices.

In [ ]:
def remove_outliers(prices) -> np.ndarray:
    """Replace values more than 3 std from the mean with the median of the valid values."""
    x = np.asarray(prices, dtype=float).ravel()
    mu, sigma = x.mean(), x.std()
    is_outlier = np.abs(x - mu) > 3 * sigma
    cleaned = x.copy()
    cleaned[is_outlier] = np.median(x[~is_outlier])
    return cleaned

In [ ]:
remove_outliers([3, 4, 4.5, 3.8, 4.2, 3.5, 4.1, 3.9, 4.4, 3.6, 999])

array([3.  , 4.  , 4.5 , 3.8 , 4.2 , 3.5 , 4.1 , 3.9 , 4.4 , 3.6 , 3.95])

## Task 3.2 Refund ratio
Refund ratio = refunds / (sales + 1e-6). The small epsilon avoids division by zero.

In [ ]:
def compute_refund_ratio(sales_count, refund_count) -> np.ndarray:
    """Element-wise refund ratio."""
    eps = 1e-6
    return np.asarray(refund_count, dtype=float) / (np.asarray(sales_count, dtype=float) + eps)

In [ ]:
compute_refund_ratio(np.array([10, 5, 0]), np.array([2, 1, 0]))

array([0.19999998, 0.19999996, 0.        ])

## Task 4.1 - Convert into pandas.DataFrame

In [ ]:
def build_dataframe(records: list[dict]) -> pd.DataFrame:
    """4.1: List of dicts -> DataFrame with datetime index, sorted chronologically."""
    df = pd.DataFrame(records)
    df["timestamp"] = pd.to_datetime(df["timestamp"])
    df = df.set_index("timestamp").sort_index()
    return df

## Task 4.2 - Calculate total price

In [ ]:
def hourly_revenue(df: pd.DataFrame) -> pd.Series:
    """4.2: Total price per item in 1-hour windows.
    Returns a Series with a MultiIndex (item, timestamp).
    """
    return df.groupby("item")["price"].resample("1h").sum()

## Task 4.3 - Total transaction revenue

In [ ]:
STATUSES = ["SUCCESS", "REFUND", "CANCELLED"]

def status_pivot_table(df: pd.DataFrame) -> pd.DataFrame:
    """4.3: Total revenue per item (rows) across status (columns)."""
    pivot = df.pivot_table(
        index="item",
        columns="status",
        values="price",
        aggfunc="sum",
        fill_value=0,
    )
    # Make sure all three status columns exist, in a fixed order
    return pivot.reindex(columns=STATUSES, fill_value=0)

## Task 4.4 - Calculate the refund ratio per item

In [ ]:
def item_refund_ratios(df: pd.DataFrame) -> pd.DataFrame:
    """4.4: Refund ratio per item, using compute_refund_ratio from Task 3."""
    counts = (
        pd.crosstab(df["item"], df["status"])
        .reindex(columns=STATUSES, fill_value=0)
    )
    sales_count = counts["SUCCESS"].to_numpy()
    refund_count = counts["REFUND"].to_numpy()
 
    ratios = compute_refund_ratio(sales_count, refund_count)
    return pd.DataFrame({"refund_ratio": ratios}, index=counts.index)

## Task 5 - Full workflow

In [ ]:
def run_pipeline(data_dir: Path = Path("data"), output_dir: Path = Path(".")) -> None:
    data_dir = Path(data_dir)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    # 1. Find log files and active register IDs
    log_files = find_log_files(data_dir)
    with open(data_dir / "store_config.json", encoding="utf-8") as f:
        config = json.load(f)
    active_ids = find_active_registers(config)
    print(f"Found {len(log_files)} log files, active registers: {sorted(active_ids)}")

    # 2. Ingest and parse
    records = []
    for path in log_files:
        text, encoding = read_file(path)
        parsed = parse_log(text)
        print(f"{path}: {len(parsed)} records ({encoding})")
        records.extend(parsed)

    # Keep only transactions from active registers
    records = [r for r in records if r["register_id"] in active_ids]

    # 3. Build DataFrame
    df = build_dataframe(records)

    # 4. Remove 999.00 outliers (NumPy)
    df["price"] = remove_outliers(df["price"])

    # 5. Currency correction: NOK prices were registered on EUR scale
    df.loc[df["currency"] == "NOK", "price"] *= 10.0

    # 6. Reports
    hr = hourly_revenue(df)
    pivot = status_pivot_table(df)
    ratios = item_refund_ratios(df)

    # 7. Save outputs
    df.to_csv(output_dir / "cleaned_transactions.csv")

    hr_df = hr.reset_index()
    hr_df["timestamp"] = hr_df["timestamp"].astype(str)
    hourly_json = {
        item: dict(zip(g["timestamp"], g["price"]))
        for item, g in hr_df.groupby("item")
    }
    with open(output_dir / "hourly_revenue.json", "w", encoding="utf-8") as f:
        json.dump(hourly_json, f, indent=2)

    pivot.to_csv(output_dir / "status_pivot_table.csv")
    ratios.to_csv(output_dir / "item_refund_ratios.csv")

    print("Done. Saved 4 output files to", output_dir.resolve())


if __name__ == "__main__":
    run_pipeline()